## Παραδοτέο 2

### Εκφώνηση:
Ανακατασκευή του συνόλου των 2 κειμένων με χρήση 3 διαφορετικών αυτόματων βιβλιοθηκών Python pipelines.

---

### Κείμενο 1:
Today is our Dragon Boat Festival, in our Chinese culture, to celebrate it with all safety and greatness in our lives. Hope you too, to enjoy it as my deepest wishes.  
Thank you for your message to show our words to the doctor, as his next contract checking, to all of us.  
I got this message to see the approved message. In fact, I have received the message from the professor, to show me this, a couple of days ago. I greatly appreciate the full support of the professor for our Springer proceedings publication.

---

### Κείμενο 2:
During our final discussion, I told him about the new submission — the one we were waiting for since last autumn, but the updates were confusing as they did not include the full feedback from the reviewer or maybe the editor?  
Anyway, I believe the team, although there was a bit of delay and less communication in recent days, really tried their best for the paper and cooperation. We should be grateful, I mean all of us, for the acceptance and efforts until the Springer link finally came last week, I think.  
Also, kindly remind me, please, if the doctor still plans to edit the acknowledgments section before sending it again. Because I didn’t see that part finalized yet, or maybe I missed it. I apologize if so.  
Overall, let us make sure all are safe and celebrate the outcome with strong coffee and future targets.

---

### Μοντέλα που χρησιμοποιήθηκαν:

1. **Πρώτο μοντέλο: Vamsi/T5_Paraphrase_Paws**  
   [Vamsi/T5_Paraphrase_Paws - Hugging Face](https://huggingface.co/Vamsi/T5_Paraphrase_Paws)  

2. **Δεύτερο μοντέλο: ramsrigouthamg/t5_paraphraser**  
   [ramsrigouthamg/t5_paraphraser - Hugging Face](https://huggingface.co/ramsrigouthamg/t5_paraphraser)  

3. **Τρίτο μοντέλο: prithivida/grammar_error_correcter_v1**  
   [prithivida/grammar_error_correcter_v1 - Hugging Face](https://huggingface.co/prithivida/grammar_error_correcter_v1)  
   Το συγκεκριμένο μοντέλο χρησιμοποιήθηκε για τη διόρθωση γραμματικών και ορθογραφικών λαθών.

4. **Τεταρτο μοντέλο: google-bert**
   [google-bert](https://huggingface.co/google-bert/bert-base-uncased)**  


In [ ]:
"""pip install tiktoken sentencepiece"""
"""!pip install tiktoken sentencepiece huggingface_hub"""
                                                                                            #test code
"""from huggingface_hub import login
# Paste in a token you get from https://huggingface.co/settings/tokens
login(token="")  """
#!pip install transformers torch scikit-learn


In [ ]:
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM, AutoModel
import torch
from sklearn.metrics.pairwise import cosine_similarity

#Models
models = [
    "Vamsi/T5_Paraphrase_Paws",
    "ramsrigouthamg/t5_paraphraser",
    "prithivida/grammar_error_correcter_v1"
]

texts = {
    "text1": (
        "Today is our dragon boat festival, in our Chinese culture, "
        "to celebrate it with all safe and great in our lives. "
        "Hope you too, to enjoy it as my deepest wishes. "
        "Thank your message to show our words to the doctor, as his next contract checking, to all of us. "
        "I got this message to see the approved message. In fact, I have received the message from the "
        "professor, to show me this a couple of days ago. I am very appreciated the full support of the "
        "professor, for our Springer proceedings publication."
    ),
    "text2": (
        "During our final discuss, I told him about the new submission — the one we were waiting since "
        "last autumn, but the updates was confusing as it not included the full feedback from reviewer or "
        "maybe editor? Anyway, I believe the team, although bit delay and less communication at recent days, they really "
        "tried best for paper and cooperation. We should be grateful, I mean all of us, for the acceptance "
        "and efforts until the Springer link came finally last week, I think. "
        "Also, kindly remind me please, if the doctor still plan for the acknowledgments section edit before "
        "he sending again. Because I didn’t see that part final yet, or maybe I missed, I apologize if so. "
        "Overall, let us make sure all are safe and celebrate the outcome with strong coffee and future targets."
    )
}

device = 0 if torch.cuda.is_available() else -1

#BERT Setup for Similarity
bert_tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
bert_model     = AutoModel.from_pretrained('bert-base-uncased')

def embed_text(text: str):
    inputs = bert_tokenizer(text, return_tensors='pt', truncation=True, padding=True)
    with torch.no_grad():
        last_hidden = bert_model(**inputs).last_hidden_state  # (1, seq_len, hidden_dim)
    mask = inputs['attention_mask'].unsqueeze(-1).expand(last_hidden.size()).float()
    summed = torch.sum(last_hidden * mask, dim=1)          # (1, hidden_dim)
    counts = mask.sum(dim=1).clamp(min=1e-9)               # avoid divide-by-zero
    return (summed / counts).squeeze(0)                    # (hidden_dim,)

def bert_cosine(a: torch.Tensor, b: torch.Tensor):
    a_np = a.unsqueeze(0).cpu().numpy()
    b_np = b.unsqueeze(0).cpu().numpy()
    return float(cosine_similarity(a_np, b_np)[0][0])

# Paraphrase + Similarity Loop 
for text_name, text in texts.items():
    print(f"\n\n######## {text_name.upper()} ########\n")
    orig_emb = embed_text(text)
    
    for model_name in models:
        print(f"-- Model: {model_name} --")
        
        tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)
        model     = AutoModelForSeq2SeqLM.from_pretrained(model_name)
        paraphraser = pipeline(
            "text2text-generation",
            model=model,
            tokenizer=tokenizer,
            device=device
        )
        
        # generate variants
        outputs = paraphraser(
            text,
            max_length=2560,
            num_return_sequences=3,
            do_sample=True,
            top_k=50,
            top_p=0.95
        )
        
        # print
        for i, out in enumerate(outputs, start=1):
            para = out['generated_text']
            sim = bert_cosine(orig_emb, embed_text(para))
            print(f"Variant {i}  –  BERT Cosine: {sim:.4f}\n{para}\n")
        print()  

## Αποτελέσματα

---

### Κείμενο 1:

Today is our Dragon Boat Festival, in our Chinese culture, to celebrate it with all safety and greatness in our lives. Hope you too, to enjoy it as my deepest wishes.  
Thank you for your message to show our words to the doctor, as his next contract checking, to all of us.  
I got this message to see the approved message. In fact, I have received the message from the professor, to show me this, a couple of days ago. I greatly appreciate the full support of the professor for our Springer proceedings publication.

---

#### Μοντέλο: Vamsi/T5_Paraphrase_Paws

- **Variant 1:**  
  Today is our Dragon Boat Festival in Chinese culture to celebrate our words with all safe and great in our lives. I have also received the message from the professor to show me this a few days ago to our Springer proceedings publication.

- **Variant 2:**  
  Today is our dragon boat festival in our Chinese culture to celebrate it with all safe and great in our lives. Hope you too, enjoy it as my deepest wishes. I received your message to see the approved message, in fact, I received the message from the professor to show me this a couple of days ago. I am very appreciated by the professor for our Springer proceedings publication.

- **Variant 3:**  
  Tonight is our Dragon Boat Festival in our Chinese culture to celebrate it with all safe and great in our lives, and I have received the letter from the professor a few days ago to show me this.

---

#### Μοντέλο: ramsrigouthamg/t5_paraphraser

- **Variant 1:**  
  Today is our dragon boat festival, in our Chinese culture, to celebrate it with all safe and great in our lives. Hope you too, to enjoy it as my deepest wishes. I got this message to see the approved message. I have received the message from the professor, to show me, this, a couple of days ago. I am very appreciated the full support of the professor, for our Springer proceedings publication.

- **Variant 2:**  
  To celebrate our dragon boat festival in our Chinese culture, to celebrate it with all safe and great in our lives. Hope you too, to enjoy it as my deepest wishes. In fact, I received the message from the professor, to show me, this, a couple of days ago. I am very appreciated the full support of the professor, for our Springer proceedings publication.

- **Variant 3:**  
  I got this message to see our words to the doctor, as his next contract checking, to all of us. Hope you too, to enjoy it with all safe and great in our lives. Thank your message to tell my words to the doctor, as his next contract checking, to all of us. I have received this message to see the approved message, in fact, I have received the message from the professor, to show me this, a couple of days ago.

---

#### Μοντέλο: prithivida/grammar_error_correcter_v1

- **Variant 1:**  
  We have honored our wishes to celebrate our dragon boat festival, in our Chinese culture, and to celebrate it with all its safety and greatness in our lives. Hope you too, to enjoy it as my deepest wishes. Thanks to your message to show our words to the doctor, for his next contract checking.

- **Variant 2:**  
  In fact, I received the message from the professor, showing me this, a couple of days ago. I am very grateful for the full support of the professor, for our Springer proceedings publication.

- **Variant 3:**  
  I have received the message from the professor, to let us know about the approval, and to show that the proposal is accepted. I have received the message, to show this, a couple of days ago. I am very appreciated the full support of the professor, for our Springer proceedings publication. Certainly.

---

### Κείμενο 2:

During our final discussion, I told him about the new submission — the one we were waiting for since last autumn, but the updates were confusing as they did not include the full feedback from the reviewer or maybe the editor?  
Anyway, I believe the team, although there was a bit of delay and less communication in recent days, really tried their best for the paper and cooperation. We should be grateful, I mean all of us, for the acceptance and efforts until the Springer link finally came last week, I think.  
Also, kindly remind me, please, if the doctor still plans to edit the acknowledgments section before sending it again. Because I didn’t see that part finalized yet, or maybe I missed it. I apologize if so.  
Overall, let us make sure all are safe and celebrate the outcome with strong coffee and future targets.

---

#### Μοντέλο: Vamsi/T5_Paraphrase_Paws

- **Variant 1:**  
  I told him about the new submission during the last discussion — the one we were waiting for since last autumn , but the updates was confusing as it did not contain the full feedback from reviewer or perhaps editor? Anyway I think the team should be grateful for acceptance and efforts till the Springer link finally came last week , I think .

- **Variant 2:**  
  I told him during the final discussion about the new submission — the one we were waiting for since autumn, but the updates was confusing as it did not include full feedback of reviewer or maybe Editor ? ?

- **Variant 3:**  
  However, in our final discussion , I told him about the new submission — the one we had been waiting for since last autumn but the updates was confusing as it was not including the full feedback from the reviewer or maybe the editor? ... Anyway , we should be grateful to the team for accepting and efforts until the Springer Link finally arrived last week, I think. Also, kindly remind me if the doctor still plans for the acknowledgments section edit before sending again . Because I didn’t see that part yet or maybe I

---

#### Μοντέλο: ramsrigouthamg/t5_paraphraser

- **Variant 1:**  
  During our final discuss, I told him about the new submission — the one we were waiting since last autumn, but the updates was confusing as it not included the full feedback from reviewer or maybe editor? Anyway, I believe the team, although bit delay and less communication at recent days, they really tried best for paper and cooperation. We should be grateful, I mean all of us, for the acceptance and efforts until the Springer Link came finally last week, I think. Also, kindly remind me please, if the doctor still plan for

- **Variant 2:**  
  I told him about the new submission — the one we were waiting since last autumn, but the updates was confusing as it not included the full feedback from reviewer or maybe editor? Anyway, I believe the team, although bit delay and less communication at recent days, they really tried best for paper and cooperation. We should be grateful, I mean all of us, for acceptance and efforts until the Springer link came finally last week, I think.

- **Variant 3:**  
  What we should be grateful for the acceptance and efforts until the Springer link came finally last week, I think. And kindly remind me please, if the doctor still plan for the acknowledgements section edit before he sending again. Because I didn’t see that part final yet, or maybe I missed, I apologize if so. Overall, let us make sure all are safe and celebrate the outcome with strong coffee and future targets.

---

#### Μοντέλο: prithivida/grammar_error_correcter_v1

- **Variant 1:**  
  At our final discuss, I told him about the new submission — the one we had been waiting for since last autumn, but the updates were confusing, as it did not include the full feedback from reviewers or maybe editors? Anyway, I believe the team, although we didn’t see that part final yet

- **Variant 2:**  
  Anyway, I believe the team, although we were waiting for the Springer link to be approved, I told him about the new submission — the one we had been waiting since last autumn, but the updates were confusing as they didn’t receive the full feedback from reviewers or maybe editors? Anyway, I believe

- **Variant 3:**  
  The last day of our final discuss, I told him about the new submission — the one we had been waiting for since last autumn, but the updates were confusing as it did not include the full feedback from reviewers or maybe editors. Anyway, I believe the team, although a bit of delay and les
